# Monitoring a deployed system for drift under active annotation

**`AsymptoticActivePPRM`** and its label-only counterpart **`AsymptoticIPWClassicalRM`** watch a quality metric across successive batches of production data when the human annotation budget of each batch is spent where your LLM judge is least reliable. They raise an alarm the moment there is statistically valid evidence that the metric has drifted past a threshold you fix in advance. This is necessary because a deployed system's quality is rarely static: traffic shifts and upstream models change, so a metric that looked fine at launch can drift silently, and a single-shot estimate made at first deployment time cannot catch that.

---

**What you will learn:**

- Why re-checking a fresh confidence interval multiple times inflates the false-alarm rate
- How `AsymptoticActivePPRM` and `AsymptoticIPWClassicalRM` give an anytime-valid alternative that stays safe under repeated checks on actively sampled labels
- How adding proxy labels lets `AsymptoticActivePPRM` detect a drift sooner than the label-only `AsymptoticIPWClassicalRM`, for the same annotation budget
- How to read their running-mean curves, confidence bounds, and alarms

## The problem: the metric can drift after launch, and repeated checking breaks validity

Let's continue with the customer-facing AI assistant of the [ASI tutorial](../../../tutorials/estimators/asi/), where hallucinations are a real risk. Every week, a fresh batch of conversations is judged by your LLM judge, which outputs a hallucination label and an **uncertainty score** for each conversation. A fixed weekly annotation budget is spent by `ActiveSampler` on the conversations where the judge is least reliable. The question is no longer "what is the hallucination rate?" but "has it drifted since launch?": your team has fixed a business threshold on the hallucination rate, and wants to know as soon as the rate has drifted past it.

The tempting approach is to compute a fresh confidence interval every week and raise an alarm the moment one crosses the threshold. This is **peeking**: each weekly test carries its own false-alarm probability, and checking after every batch compounds those chances, so a false alarm becomes likely over a long enough monitoring horizon even if nothing has actually drifted. The [Monitors user guide](../../../user_guide/monitors/) explains why this happens and how an anytime-valid confidence sequence that stays safe no matter how often you look avoids it. This tutorial focuses on what that means for you in practice, when the human labels are drawn with non-uniform probabilities.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ASIMeanEstimator
from glide.monitors import AsymptoticActivePPRM, AsymptoticIPWClassicalRM
from glide.samplers import ActiveSampler
from glide.simulators import generate_batched_binary_dataset_with_oracle_sampling, simulate_annotation

# ── Colour palette ──────────────────────────────────────────
C_ESTIMATE = "#95A5A6"  # per-batch lower bound               — grey
C_NAIVE = "#E74C3C"  # naive baseline alarm                   — red-orange
C_ACTIVE_PPRM = "#27AE60"  # Active PPRM running bound        — green
C_IPW_CLASSICAL = "#2980B9"  # IPW classical running bound    — blue
C_ALARM_ACTIVE_PPRM = "#8E44AD"  # Active PPRM alarm          — purple
C_ALARM_IPW_CLASSICAL = "#F39C12"  # IPW classical alarm      — gold
C_THRESHOLD = "#2C3E50"  # threshold                          — dark slate

# ── Global plot style ─────────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 14,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

## Simulating a batched, actively sampled production stream

We use `generate_batched_binary_dataset_with_oracle_sampling` to simulate successive batches of conversations, each with its own true hallucination rate, proxy rate, and correlation. It returns the ground-truth labels, the judge's labels, the judge's uncertainty scores, and an integer identifying the chronological batch each row belongs to.

`simulate_batches` below builds such a stream from a list of per-week true hallucination rates. Every week gets `BATCH_SIZE` proxy-judged conversations, of which about `LABELS_PER_BATCH` are sent for human review. We call `ActiveSampler` **once per week**: each week has its own annotation budget and its own uncertainty scores, so its sampling probabilities must be computed from that week's conversations only. The per-week sampling probabilities and selection indicators are then stacked, and `simulate_annotation` reveals the human labels of the selected conversations once over the whole stream, leaving the rest as `np.nan`.

The monitors need the sampling probabilities to correct for the fact that uncertain conversations are over-represented among the annotated ones. We fix an alarm threshold above the stationary rate and assess the monitoring methods' ability to detect when it is exceeded while avoiding false alarms. All batches in the stream are monitored.

In [ ]:
N_BATCHES = 50
BATCH_SIZE = 800
LABELS_PER_BATCH = 40
STATIONARY_MEAN = 0.28
PROXY_BIAS = 0.05
CORRELATION = 0.85
CONFIDENCE_LEVEL = 0.8
THRESHOLD = 0.3


def simulate_batches(true_means, random_seed):
    proxy_means = true_means + PROXY_BIAS
    correlations = [CORRELATION] * len(true_means)
    y_true_oracle, y_proxy, uncertainty, batches = generate_batched_binary_dataset_with_oracle_sampling(
        n_samples=[BATCH_SIZE] * len(true_means),
        true_mean=true_means,
        proxy_mean=proxy_means,
        correlation=correlations,
        random_seed=random_seed,
    )
    per_batch_samples = [
        ActiveSampler().sample(
            uncertainty[batches == batch_id], n_samples=LABELS_PER_BATCH, random_seed=random_seed + batch_id
        )
        for batch_id in range(N_BATCHES)
    ]
    sampling_probability = np.hstack([probabilities for probabilities, _ in per_batch_samples])
    xi = np.hstack([indicators for _, indicators in per_batch_samples])
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, sampling_probability, batches

## The naive baseline: one confidence interval per batch

The tempting-but-invalid approach simply calls `ASIMeanEstimator` once per week, comparing that week's confidence interval to the threshold and alarming whenever the interval lies above it. It keeps the proxy labels and the active sampling, and only drops the anytime validity.

In [ ]:
def naive_per_batch_alarms(y_true, y_proxy, sampling_probability, batches, threshold, n_batches):
    means = np.zeros(n_batches)
    lower_bounds = np.zeros(n_batches)
    alarms = np.zeros(n_batches, dtype=bool)
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        batch_result = ASIMeanEstimator().estimate(
            y_true[batch_mask],
            y_proxy[batch_mask],
            sampling_probability[batch_mask],
            confidence_level=CONFIDENCE_LEVEL,
        )
        means[batch_id] = batch_result.mean
        lower_bounds[batch_id] = batch_result.confidence_interval.lower_bound
        alarms[batch_id] = batch_result.confidence_interval.lower_bound > threshold
    return means, lower_bounds, alarms

In [ ]:
def plot_monitor_vs_baseline(lower_bounds, active_pprm_result, ipw_classical_result, naive_alarms, title):
    weeks = np.arange(1, len(lower_bounds) + 1)
    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.scatter(weeks, lower_bounds, color=C_ESTIMATE, s=28, alpha=0.8, zorder=3, label="Per-batch lower bounds")
    ax.plot(
        weeks,
        active_pprm_result.confidence_bounds,
        color=C_ACTIVE_PPRM,
        linewidth=2.5,
        zorder=4,
        label="Active PPRM lower bounds",
    )
    ax.plot(
        weeks,
        ipw_classical_result.confidence_bounds,
        color=C_IPW_CLASSICAL,
        linewidth=2.5,
        zorder=4,
        label="IPW classical monitor lower bounds",
    )
    ax.axhline(THRESHOLD, color=C_THRESHOLD, linestyle="--", linewidth=1.8, zorder=2, label="Threshold")

    if naive_alarms.any():
        ax.scatter(
            weeks[naive_alarms],
            lower_bounds[naive_alarms],
            marker="+",
            s=110,
            linewidths=2.5,
            color=C_NAIVE,
            zorder=5,
            label="Naive baseline alarms",
        )

    if active_pprm_result.drift_detected:
        alarm_week = active_pprm_result.first_alarm_index + 1
        ax.axvline(
            alarm_week,
            color=C_ALARM_ACTIVE_PPRM,
            linestyle=":",
            linewidth=2.2,
            zorder=2,
            label=f"Active PPRM alarm (week {alarm_week})",
        )

    if ipw_classical_result.drift_detected:
        alarm_week = ipw_classical_result.first_alarm_index + 1
        ax.axvline(
            alarm_week,
            color=C_ALARM_IPW_CLASSICAL,
            linestyle=":",
            linewidth=2.2,
            zorder=2,
            label=f"IPW classical alarm (week {alarm_week})",
        )

    ax.set_xlabel("Week")
    ax.set_ylabel("Hallucination Rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", framealpha=0.9)
    plt.tight_layout()
    plt.show()

## Scenario 1: no drift, no false alarm

We generate a fully stationary stream: the true hallucination rate stays below the threshold. Neither monitor should alarm, except by chance.

In [ ]:
true_means_no_drift = np.full(N_BATCHES, STATIONARY_MEAN)
y_true_no_drift, y_proxy_no_drift, sampling_probability_no_drift, batches_no_drift = simulate_batches(
    true_means_no_drift, random_seed=7
)

n_labeled = np.sum(~np.isnan(y_true_no_drift))
print(f"Total conversations : {len(y_true_no_drift):,}")
print(f"  Human-reviewed     : {n_labeled:,}")
print(f"  Number of weeks    : {len(np.unique(batches_no_drift))}")

In [ ]:
n_nan_in_y_true = np.isnan(y_true_no_drift).sum()
n_zeros_in_y_true = np.sum(y_true_no_drift == 0)
n_ones_in_y_true = np.sum(y_true_no_drift == 1)
print("Analysis of the human-reviewed labels:")
print(f"Number of NaNs: {n_nan_in_y_true}")
print(f"Number of 0s: {n_zeros_in_y_true}")
print(f"Number of 1s: {n_ones_in_y_true}")

In [ ]:
print("Analysis of the sampling probabilities:")
print(f"Minimum: {sampling_probability_no_drift.min():.3f}")
print(f"Mean: {sampling_probability_no_drift.mean():.3f}")
print(f"Maximum: {sampling_probability_no_drift.max():.3f}")
print(f"Uniform probability (for reference): {LABELS_PER_BATCH / BATCH_SIZE:.3f}")

Run the naive baseline and both monitoring methods. `AsymptoticActivePPRM` takes the proxy labels and the sampling probabilities on top of the human labels, while `AsymptoticIPWClassicalRM` takes the human labels and the sampling probabilities only. The label-only counterpart has to be the inverse-probability-weighted one: a plain `AsymptoticClassicalRM` would average the actively sampled labels as if they were drawn uniformly, and would therefore be biased towards the conversations the judge is least sure about.

In [ ]:
means_no_drift, lower_no_drift, naive_alarms_no_drift = naive_per_batch_alarms(
    y_true_no_drift,
    y_proxy_no_drift,
    sampling_probability_no_drift,
    batches_no_drift,
    THRESHOLD,
    N_BATCHES,
)
active_pprm_result_no_drift = AsymptoticActivePPRM().detect(
    y_true_no_drift,
    y_proxy_no_drift,
    sampling_probability_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
ipw_classical_result_no_drift = AsymptoticIPWClassicalRM().detect(
    y_true_no_drift,
    sampling_probability_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
naive_alarm_weeks = np.arange(1, N_BATCHES + 1)[naive_alarms_no_drift]

print(f"Naive baseline alarms at week(s): {naive_alarm_weeks}")
print(f"Active PPRM drift detected: {active_pprm_result_no_drift.drift_detected}")
print(f"IPW classical monitor drift detected: {ipw_classical_result_no_drift.drift_detected}")
print(
    f"Active PPRM running mean / bound after {N_BATCHES} weeks: {active_pprm_result_no_drift.running_means[-1]:.1%} / "
    f"{active_pprm_result_no_drift.confidence_bounds[-1]:.1%}"
)
print(
    f"IPW classical monitor running mean / bound after {N_BATCHES} weeks: "
    f"{ipw_classical_result_no_drift.running_means[-1]:.1%} / {ipw_classical_result_no_drift.confidence_bounds[-1]:.1%}"
)

In [ ]:
plot_monitor_vs_baseline(
    lower_no_drift,
    active_pprm_result_no_drift,
    ipw_classical_result_no_drift,
    naive_alarms_no_drift,
    title="Scenario 1: stationary stream (no drift)",
)

At some point, ordinary sampling noise pushes one of the confidence intervals above the threshold, and the naive baseline raises false alarms, purely from **peeking**: with a fresh, independent-looking test every week, the chance of at least one false alarm climbs the longer you keep checking, even though the hallucination rate did not move. `AsymptoticActivePPRM` and `AsymptoticIPWClassicalRM`, in contrast, never alarm across the full stream: their running means stay below the threshold, with anytime-valid lower bounds safely underneath it as well.

## Scenario 2: slow drift, early alarm

Now we generate data with a progressive drift: partway through the stream, the hallucination rate ramps up to an unacceptably high level. The judge's bias and its correlation with the human labels stay the same, so the drift only moves the true hallucination rate.

In [ ]:
DRIFT_START = 15
DRIFT_AMPLITUDE = 0.35
RAMP_LENGTH = 30

true_means_slow_drift = np.hstack(
    [
        np.full(DRIFT_START, STATIONARY_MEAN),
        np.linspace(STATIONARY_MEAN, STATIONARY_MEAN + DRIFT_AMPLITUDE, RAMP_LENGTH),
        np.full(N_BATCHES - RAMP_LENGTH - DRIFT_START, STATIONARY_MEAN + DRIFT_AMPLITUDE),
    ]
)
y_true_slow_drift, y_proxy_slow_drift, sampling_probability_slow_drift, batches_slow_drift = simulate_batches(
    true_means_slow_drift, random_seed=0
)

means_slow_drift, lower_slow_drift, naive_alarms_slow_drift = naive_per_batch_alarms(
    y_true_slow_drift,
    y_proxy_slow_drift,
    sampling_probability_slow_drift,
    batches_slow_drift,
    THRESHOLD,
    N_BATCHES,
)
active_pprm_result_slow_drift = AsymptoticActivePPRM().detect(
    y_true_slow_drift,
    y_proxy_slow_drift,
    sampling_probability_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
ipw_classical_result_slow_drift = AsymptoticIPWClassicalRM().detect(
    y_true_slow_drift,
    sampling_probability_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)

naive_alarm_weeks = np.flatnonzero(naive_alarms_slow_drift) + 1
print(f"Naive baseline alarms: {len(naive_alarm_weeks)} weeks, first at week {naive_alarm_weeks[0]}")
active_pprm_alarm_index = active_pprm_result_slow_drift.first_alarm_index
if active_pprm_alarm_index is not None:
    print(f"Active PPRM drift detected: True, first alarm at week {active_pprm_alarm_index + 1}")
ipw_classical_alarm_index = ipw_classical_result_slow_drift.first_alarm_index
if ipw_classical_alarm_index is not None:
    print(f"IPW classical monitor drift detected: True, first alarm at week {ipw_classical_alarm_index + 1}")

In [ ]:
plot_monitor_vs_baseline(
    lower_slow_drift,
    active_pprm_result_slow_drift,
    ipw_classical_result_slow_drift,
    naive_alarms_slow_drift,
    title="Scenario 2: drift occurrence during deployment",
)

Observe that the peeking baseline alarms earlier. However, this occurs even before the drift happens and, therefore, qualifies as a false alarm. In real time, a team watching the naive baseline has no way to tell a validated alarm from a lucky (or unlucky) one, because its error rate was never controlled for repeated testing in the first place: Scenario 1 already showed it firing on a completely stationary stream. On the other hand, the anytime-valid alarms come fairly early after a consistent drift can be observed and not before, which makes them more trustworthy. Between the two, `AsymptoticActivePPRM` alarms sooner than `AsymptoticIPWClassicalRM`: both monitors spend the same weekly human budget on the same conversations, but the correlated proxy labels sharpen the per-batch estimate of `AsymptoticActivePPRM`, shrinking its confidence sequence and letting it accumulate evidence of the drift faster.

`AsymptoticActivePPRM` and `AsymptoticIPWClassicalRM`'s alarms carry no ambiguity: each is backed by the same anytime-valid guarantee demonstrated in Scenario 1, so under no drift the chance either would ever fire at all across the whole monitoring horizon corresponds to the used confidence level. At the moment each fires, its anytime-valid bound has just crossed the threshold. By the end of the stream, both running means and their bounds are clearly above the threshold.

## Reading the result object

In both scenarios, `AsymptoticActivePPRM` returned a `PredictionPoweredMeanMonitoringResult`. Its public surface covers everything used above, plus a formatted summary. Note that the number of proxy labels per batch counts the conversations that entered the estimate, i.e. those with a strictly positive sampling probability.

For the full derivation of the quantities behind this result, see the [Asymptotic Active PPRM section of the Monitors user guide](../../../user_guide/monitors/#asymptotic-active-prediction-powered-risk-monitoring-asymptotic-active-pprm).

In [ ]:
print("Active PPRM")
print(f"{'Drift detected':<32}: {active_pprm_result_slow_drift.drift_detected}")
print(f"{'First alarm index':<32}: {active_pprm_result_slow_drift.first_alarm_index}")
print(f"{'Alarm threshold':<32}: {active_pprm_result_slow_drift.alarm_threshold}")
print(f"{'Confidence level':<32}: {active_pprm_result_slow_drift.confidence_level}")
print(f"{'Running means (last 3)':<32}: {np.round(active_pprm_result_slow_drift.running_means[-3:], 3)}")
print(f"{'Confidence bounds (last 3)':<32}: {np.round(active_pprm_result_slow_drift.confidence_bounds[-3:], 3)}")
print(f"{'True labels per batch (last 3)':<32}: {active_pprm_result_slow_drift.batch_n_true[-3:]}")
print(f"{'Proxy labels per batch (last 3)':<32}: {active_pprm_result_slow_drift.batch_n_proxy[-3:]}")

## Summary : Anytime-valid Risk Monitoring detects drifts and reduces false alarms under active annotation

| | Per-batch baseline | `AsymptoticIPWClassicalRM` | `AsymptoticActivePPRM` |
|-|---------------------|-------------------|-------------------|
| Valid under repeated looks? | ❌ No | ✅ Yes | ✅ Yes |
| Uses proxy labels? | ✅ Yes | ❌ No | ✅ Yes |
| Detects a sustained drift? | 🟠 Yes, but prone to early false alarms | ✅ Yes | ✅ Yes, sooner than the IPW classical monitor |

**Key takeaways:**

1. **Repeated significance testing is not a monitor.** Scenario 1 showed the per-batch baseline raise a false alarm on a perfectly stationary stream, purely from checking every week, so its alarms carry no error-rate guarantee, correct or not.
2. **An anytime-valid confidence sequence makes checking every week safe.** Both `AsymptoticActivePPRM` and `AsymptoticIPWClassicalRM`'s guarantees hold simultaneously across all looks, so their alarms in Scenario 2 can be trusted the moment they fire, without waiting to see whether later weeks confirm them.
3. **Active sampling plus a proxy makes the monitor more sensitive for the same budget.** `AsymptoticActivePPRM` alarmed sooner than `AsymptoticIPWClassicalRM` in Scenario 2: both monitors rely on the same uncertainty-guided human labels, and the correlated proxy labels sharpen the confidence sequence on top of them.

---

*Want to go further? The [Asymptotic PPRM scientific validation notebook](../../../deep_dive/scientific_validation/monitors/asymptotic_pprm/) provides rigorous empirical evidence on false-alarm control and detection power, the [Monitors user guide](../../../user_guide/monitors/) derives the anytime-valid confidence sequence used here from first principles.*